# Feature Engineering — P2: Riesgo operacional

**Proyecto:** MiBici — TC5061 Equipo 02  
**Pregunta:** P2 — ¿Qué estación tendrá un alto riesgo de desequilibrio operacional durante la siguiente hora?

Este notebook documenta y evidencia el módulo reusable `mi_bici/features.py`. No entrena modelos ni realiza nuevos experimentos de ML.

## Objetivo

La unidad de predicción es **estación-hora**. Las variables de entrada representan información disponible hasta la hora `t`, mientras que el objetivo corresponde al desequilibrio operacional de la hora siguiente `t+1`.

El objetivo continuo de P2 es:

$$\[
r_{t+1} = \frac{|D_{t+1} - A_{t+1}|}{C}
\]
$$

donde:

- `D` = viajes que salen de la estación.
- `A` = viajes que llegan a la estación.
- `C` = capacidad de la estación.

La clasificación binaria `Risk=1/0` se construye posteriormente en `train.py` usando el percentil 80 **calculado únicamente sobre train**, para evitar leakage temporal.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Locate the repository root so the notebook can import the project package
# whether Jupyter is launched from the repository root or from notebooks/.
ROOT = Path.cwd()
while not (ROOT / "mi_bici").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "mi_bici").exists():
    raise FileNotFoundError(
        "No se encontró la carpeta 'mi_bici'. "
        "Abre este notebook desde el repositorio del proyecto."
    )

sys.path.insert(0, str(ROOT))

from mi_bici.config import PROCESSED_DATA_DIR
from mi_bici.features import (
    load_trips,
    load_stations,
    build_station_hour_table,
)

print("Repository root:", ROOT)
print("Processed data:", PROCESSED_DATA_DIR)


2026-10-16 11:14:44.010 | INFO     | mi_bici.config:<module>:11 - PROJ_ROOT path is: C:\Users\eduar\Downloads\TEST\TC5061-Proyecto-Equipo02


Repository root: c:\Users\eduar\Downloads\TEST\TC5061-Proyecto-Equipo02
Processed data: C:\Users\eduar\Downloads\TEST\TC5061-Proyecto-Equipo02\data\processed


## 1. Ingesta de insumos

`features.py` reutiliza los datos procesados por el pipeline de ingesta:

- `viajes_*.parquet`: viajes mensuales normalizados.
- `estaciones.parquet`: catálogo normalizado de estaciones.

Las funciones `load_trips()` y `load_stations()` encapsulan la lectura y normalización básica. Esto evita duplicar lógica de carga dentro del modelado.


In [2]:
# Las siguientes funciones son las utilizadas por el pipeline reproducible.
# Se muestran aquí para documentar la entrada de feature engineering.
#
# trips = load_trips(PROCESSED_DATA_DIR)
# stations = load_stations(PROCESSED_DATA_DIR)
#
# El procesamiento completo puede ser costoso porque reconstruye la malla
# estación-hora. El dataset ya generado se inspecciona a continuación.

features_path = PROCESSED_DATA_DIR / "features_p2.parquet"

if not features_path.is_file():
    raise FileNotFoundError(
        f"No se encontró {features_path}. "
        "Ejecuta primero: python -m mi_bici.features"
    )

features = pd.read_parquet(features_path)

print(f"Dataset P2: {len(features):,} observaciones")
print(f"Columnas: {len(features.columns)}")
features.head()


Dataset P2: 6,289,920 observaciones
Columnas: 37


,station_id,hour,departures,arrivals,station_name,capacity,latitude,longitude,net_flow,absolute_imbalance,...,arrivals_rolling_24h,net_flow_lag_1h,net_flow_lag_2h,net_flow_lag_24h,net_flow_rolling_3h,net_flow_rolling_24h,target_departures,target_arrivals,target_net_flow,target_imbalance_ratio
0,10,2025-01-01 00:00:00,0.0,0.0,(GDL-008) C. Ignacio Zaragoza / C. J. Angulo,15,20.681953,-103.349992,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
1,10,2025-01-01 01:00:00,0.0,0.0,(GDL-008) C. Ignacio Zaragoza / C. J. Angulo,15,20.681953,-103.349992,0.0,0.0,...,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
2,10,2025-01-01 02:00:00,0.0,0.0,(GDL-008) C. Ignacio Zaragoza / C. J. Angulo,15,20.681953,-103.349992,0.0,0.0,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0
3,10,2025-01-01 03:00:00,0.0,0.0,(GDL-008) C. Ignacio Zaragoza / C. J. Angulo,15,20.681953,-103.349992,0.0,0.0,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0
4,10,2025-01-01 04:00:00,0.0,0.0,(GDL-008) C. Ignacio Zaragoza / C. J. Angulo,15,20.681953,-103.349992,0.0,0.0,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Construcción de la unidad estación-hora

`build_station_hour_table()` transforma los viajes individuales en una observación por:

**estación × hora**

Primero se calculan por separado:

- `departures`: viajes que parten de la estación durante la hora.
- `arrivals`: viajes que llegan a la estación durante la hora.

Después se construye una malla horaria completa para las estaciones y se rellenan con `0` las horas sin actividad observada. Finalmente se incorporan los metadatos de estación.

Esto permite trabajar con una serie temporal regular por estación.


In [3]:
print("Rango temporal:")
print("  Inicio:", features["hour"].min())
print("  Fin:   ", features["hour"].max())

print("\nNúmero de estaciones:", features["station_id"].nunique())
print("Observaciones:", f"{len(features):,}")

print("\nEjemplo de observaciones:")
display(
    features[
        [
            "station_id",
            "hour",
            "departures",
            "arrivals",
            "capacity",
        ]
    ].head(10)
)


Rango temporal:
  Inicio: 2025-01-01 00:00:00
  Fin:    2026-06-30 23:00:00

Número de estaciones: 480
Observaciones: 6,289,920

Ejemplo de observaciones:


,station_id,hour,departures,arrivals,capacity
0,10,2025-01-01 00:00:00,0.0,0.0,15
1,10,2025-01-01 01:00:00,0.0,0.0,15
2,10,2025-01-01 02:00:00,0.0,0.0,15
3,10,2025-01-01 03:00:00,0.0,0.0,15
4,10,2025-01-01 04:00:00,0.0,0.0,15
5,10,2025-01-01 05:00:00,0.0,0.0,15
6,10,2025-01-01 06:00:00,0.0,0.0,15
7,10,2025-01-01 07:00:00,0.0,1.0,15
8,10,2025-01-01 08:00:00,0.0,0.0,15
9,10,2025-01-01 09:00:00,1.0,1.0,15


## 3. Variables derivadas de flujo

A partir de las salidas y llegadas se generan:

- `net_flow = departures - arrivals`
- `absolute_imbalance = |net_flow|`
- `imbalance_ratio = absolute_imbalance / capacity`

Estas variables representan el estado operacional de la estación en la hora actual `t`.

**Importante:** no son el objetivo de P2. Son variables disponibles en el momento de hacer la predicción de la siguiente hora.


In [4]:
derived_cols = [
    "departures",
    "arrivals",
    "net_flow",
    "absolute_imbalance",
    "imbalance_ratio",
    "capacity",
]

display(features[derived_cols].head())

# Verify the algebra used by the reusable feature module.
assert np.allclose(
    features["net_flow"],
    features["departures"] - features["arrivals"],
    equal_nan=True,
)

assert np.allclose(
    features["absolute_imbalance"],
    features["net_flow"].abs(),
    equal_nan=True,
)

assert np.allclose(
    features["imbalance_ratio"],
    features["absolute_imbalance"] / features["capacity"],
    equal_nan=True,
)

print("✓ Relaciones de flujo verificadas.")


,departures,arrivals,net_flow,absolute_imbalance,imbalance_ratio,capacity
0,0.0,0.0,0.0,0.0,0.0,15
1,0.0,0.0,0.0,0.0,0.0,15
2,0.0,0.0,0.0,0.0,0.0,15
3,0.0,0.0,0.0,0.0,0.0,15
4,0.0,0.0,0.0,0.0,0.0,15


✓ Relaciones de flujo verificadas.


## 4. Variables temporales y cíclicas

Se crean variables para representar patrones de hora y día:

- `hour_of_day`
- `day_of_week`
- `is_weekend`
- `hour_sin`, `hour_cos`
- `dow_sin`, `dow_cos`

Las transformaciones seno/coseno representan la naturaleza cíclica del tiempo: por ejemplo, 23:00 y 00:00 son horas cercanas aunque sus valores enteros sean muy diferentes.


In [5]:
time_cols = [
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
]

display(features[["hour"] + time_cols].head())


,hour,hour_of_day,day_of_week,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos
0,2025-01-01 00:00:00,0,2,0,0.000000,1.000000,0.974928,-0.222521
1,2025-01-01 01:00:00,1,2,0,0.258819,0.965926,0.974928,-0.222521
2,2025-01-01 02:00:00,2,2,0,0.500000,0.866025,0.974928,-0.222521
3,2025-01-01 03:00:00,3,2,0,0.707107,0.707107,0.974928,-0.222521
4,2025-01-01 04:00:00,4,2,0,0.866025,0.500000,0.974928,-0.222521


## 5. Variables históricas: rezagos y ventanas móviles

Para cada estación se calculan variables históricas de:

- `departures`
- `arrivals`
- `net_flow`

Se generan:

- rezago de 1 hora (`lag_1h`)
- rezago de 2 horas (`lag_2h`)
- rezago de 24 horas (`lag_24h`)
- promedio móvil de 3 horas
- promedio móvil de 24 horas

### Prevención de leakage

Las ventanas móviles se calculan con:

```text
shift(1).rolling(...)
```

Es decir, la observación actual `t` se excluye de la ventana histórica. Por tanto, una ventana asociada a `t` utiliza únicamente información anterior a `t`.

Los rezagos utilizan `shift(1)`, `shift(2)` y `shift(24)`, por lo que tampoco incorporan información futura.


In [6]:
historical_cols = [
    "departures_lag_1h",
    "departures_lag_2h",
    "departures_lag_24h",
    "departures_rolling_3h",
    "departures_rolling_24h",
    "arrivals_lag_1h",
    "arrivals_lag_2h",
    "arrivals_lag_24h",
    "arrivals_rolling_3h",
    "arrivals_rolling_24h",
    "net_flow_lag_1h",
    "net_flow_lag_2h",
    "net_flow_lag_24h",
    "net_flow_rolling_3h",
    "net_flow_rolling_24h",
]

print(f"Variables históricas: {len(historical_cols)}")
display(features[["station_id", "hour"] + historical_cols].head())


Variables históricas: 15


,station_id,hour,departures_lag_1h,departures_lag_2h,departures_lag_24h,departures_rolling_3h,departures_rolling_24h,arrivals_lag_1h,arrivals_lag_2h,arrivals_lag_24h,arrivals_rolling_3h,arrivals_rolling_24h,net_flow_lag_1h,net_flow_lag_2h,net_flow_lag_24h,net_flow_rolling_3h,net_flow_rolling_24h
0,10,2025-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10,2025-01-01 01:00:00,0.0,NaN,NaN,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,NaN,NaN,0.0,0.0
2,10,2025-01-01 02:00:00,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,0.0
3,10,2025-01-01 03:00:00,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,0.0
4,10,2025-01-01 04:00:00,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,0.0


## 6. Objetivo de P2: siguiente hora

El objetivo se construye mediante `shift(-1)` dentro de cada estación:

```python
target_departures = departures(t+1)
target_arrivals = arrivals(t+1)
```

y posteriormente:

\[
target\_imbalance\_ratio =
\frac{|target\_departures-target\_arrivals|}{capacity}
\]

Por construcción, el objetivo corresponde a la **siguiente hora**, mientras que las variables de entrada representan el estado actual o información histórica.

Las columnas `target_*` son variables objetivo y **no forman parte de las entradas del modelo**.


In [7]:
target_cols = [
    "target_departures",
    "target_arrivals",
    "target_net_flow",
    "target_imbalance_ratio",
]

display(features[["station_id", "hour"] + target_cols].head())

# The final observation of every station cannot have a next-hour target,
# therefore it is removed by build_station_hour_table().
missing_targets = features[target_cols].isna().any(axis=1).sum()
print("Filas con target faltante:", missing_targets)

assert missing_targets == 0
print("✓ Todas las observaciones conservan un objetivo de siguiente hora.")


,station_id,hour,target_departures,target_arrivals,target_net_flow,target_imbalance_ratio
0,10,2025-01-01 00:00:00,0.0,0.0,0.0,0.0
1,10,2025-01-01 01:00:00,0.0,0.0,0.0,0.0
2,10,2025-01-01 02:00:00,0.0,0.0,0.0,0.0
3,10,2025-01-01 03:00:00,0.0,0.0,0.0,0.0
4,10,2025-01-01 04:00:00,0.0,0.0,0.0,0.0


Filas con target faltante: 0
✓ Todas las observaciones conservan un objetivo de siguiente hora.


## 7. Auditoría de leakage temporal

La separación conceptual de variables es:

| Grupo | Momento de información |
|---|---|
| Estado actual (`departures`, `arrivals`, `net_flow`, etc.) | `t` |
| Rezagos y ventanas | `< t` |
| Variables temporales | asociadas a `t` |
| Objetivo `target_*` | `t+1` |

La construcción del objetivo usa `shift(-1)`, mientras que las variables históricas usan `shift(1)`, `shift(2)`, `shift(24)` y ventanas precedidas por `shift(1)`.

El umbral binario de riesgo tampoco se calcula sobre todo el dataset: `train.py` obtiene el P80 exclusivamente del conjunto de entrenamiento.


In [8]:
# Audit that target columns are not accidentally included among model inputs.
FEATURE_COLUMNS = [
    "departures",
    "arrivals",
    "net_flow",
    "absolute_imbalance",
    "imbalance_ratio",
    "capacity",
    "latitude",
    "longitude",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "departures_lag_1h",
    "departures_lag_2h",
    "departures_lag_24h",
    "departures_rolling_3h",
    "departures_rolling_24h",
    "arrivals_lag_1h",
    "arrivals_lag_2h",
    "arrivals_lag_24h",
    "arrivals_rolling_3h",
    "arrivals_rolling_24h",
    "net_flow_lag_1h",
    "net_flow_lag_2h",
    "net_flow_lag_24h",
    "net_flow_rolling_3h",
    "net_flow_rolling_24h",
]

leakage_names = [c for c in FEATURE_COLUMNS if c.startswith("target_")]

assert len(FEATURE_COLUMNS) == 30
assert not leakage_names
assert all(c in features.columns for c in FEATURE_COLUMNS)

print("Número de variables usadas por el modelo:", len(FEATURE_COLUMNS))
print("✓ Ninguna variable target_* está incluida en FEATURE_COLUMNS.")


Número de variables usadas por el modelo: 30
✓ Ninguna variable target_* está incluida en FEATURE_COLUMNS.


## 8. Variables finales utilizadas por el modelo P2

Las 30 variables de entrada se agrupan conceptualmente así:

1. **Estado actual:** actividad y desequilibrio de la estación.
2. **Capacidad y ubicación:** capacidad, latitud y longitud.
3. **Tiempo:** hora, día de semana, fin de semana y codificaciones cíclicas.
4. **Histórico:** rezagos y ventanas móviles de salidas, llegadas y flujo neto.

Las columnas `target_departures`, `target_arrivals`, `target_net_flow` y `target_imbalance_ratio` se mantienen en el dataset para evaluación, pero no se pasan al modelo.


In [9]:
for i, col in enumerate(FEATURE_COLUMNS, start=1):
    print(f"{i:02d}. {col}")


01. departures
02. arrivals
03. net_flow
04. absolute_imbalance
05. imbalance_ratio
06. capacity
07. latitude
08. longitude
09. hour_of_day
10. day_of_week
11. is_weekend
12. hour_sin
13. hour_cos
14. dow_sin
15. dow_cos
16. departures_lag_1h
17. departures_lag_2h
18. departures_lag_24h
19. departures_rolling_3h
20. departures_rolling_24h
21. arrivals_lag_1h
22. arrivals_lag_2h
23. arrivals_lag_24h
24. arrivals_rolling_3h
25. arrivals_rolling_24h
26. net_flow_lag_1h
27. net_flow_lag_2h
28. net_flow_lag_24h
29. net_flow_rolling_3h
30. net_flow_rolling_24h


## 9. Umbral de riesgo

La variable continua `target_imbalance_ratio` se convierte en una etiqueta binaria en `train.py`.

El umbral utilizado es el **percentil 80 (P80) calculado únicamente con train**:

```text
P80 = 0.066667
```

Por lo tanto:

```text
Risk = 1  si target_imbalance_ratio >= P80_train
Risk = 0  en otro caso
```

Este P80 es un **proxy relativo de riesgo operacional**: identifica el 20% superior de desequilibrios dentro de la distribución de entrenamiento. No debe interpretarse como un umbral físico o de seguridad universal para una estación.

Esta notebook no recalcula ni modifica el umbral y no entrena modelos.


## 10. Reproducibilidad

La transformación completa está implementada en:

```text
mi_bici/features.py
```

y puede reproducirse con:

```powershell
python -m mi_bici.features
```

El archivo generado es:

```text
data/processed/features_p2.parquet
```

El entrenamiento posterior se ejecuta con:

```powershell
python -m mi_bici.modeling.train
```

La separación entre **feature engineering** y **modelado** permite reutilizar las mismas transformaciones y mantener el pipeline reproducible.


## Conclusión

El módulo de feature engineering de P2:

- trabaja a granularidad estación-hora;
- construye variables de estado actual, tiempo, ubicación e historial;
- utiliza rezagos y ventanas móviles sin incorporar información futura;
- define el objetivo en `t+1`;
- mantiene las variables objetivo separadas de las entradas del modelo;
- deja el cálculo del umbral P80 para `train.py`, donde se realiza únicamente con train.

**No se realizaron nuevos experimentos de ML en este notebook.**
